In [ ]:
"""
================================================================================
Advanced Bidirectional LSTM for Sentiment Analysis — TensorFlow / Keras
================================================================================
Production-grade pipeline covering:

  1.  Data loading — IMDB (built-in) + custom CSV ingestion
  2.  Text preprocessing — custom tokenizer, padding, OOV handling
  3.  Embedding strategies:
        a) Trainable embeddings (from scratch)
        b) Pre-trained GloVe embeddings (transfer learning)
  4.  Four model architectures:
        a) BiLSTM Baseline
        b) Stacked BiLSTM with Residual Connections
        c) BiLSTM + Multi-Head Self-Attention
        d) BiLSTM + CNN Feature Extraction (Hybrid)
  5.  Custom Self-Attention & Multi-Head Attention layers
  6.  Regularisation — spatial dropout, recurrent dropout, label smoothing
  7.  Training — mixed precision, LR scheduling, class weighting
  8.  Evaluation — accuracy, F1, precision, recall, AUC-ROC, confusion matrix
  9.  Interpretability — attention weight extraction & visualisation
  10. Inference API — predict sentiment on raw text with confidence score
================================================================================
"""

import os, re, warnings, string
from typing import Tuple, List, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, backend as K
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score,
)
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  1. DATA LOADING & TEXT PREPROCESSING                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class TextPreprocessor:
    """
    End-to-end text preprocessing pipeline:
      • HTML tag removal
      • Lowercasing & punctuation stripping
      • Tokenization with OOV token
      • Sequence padding (pre/post)
      • Vocabulary statistics
    """

    def __init__(
        self,
        max_vocab: int = 25_000,
        max_len: int = 256,
        oov_token: str = "<OOV>",
        padding: str = "post",
        truncating: str = "post",
    ):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.oov_token = oov_token
        self.padding = padding
        self.truncating = truncating
        self.tokenizer = Tokenizer(
            num_words=max_vocab, oov_token=oov_token,
            filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
        )
        self._fitted = False

    @staticmethod
    def clean_text(text: str) -> str:
        """Remove HTML, URLs, special chars; lowercase."""
        text = re.sub(r"<[^>]+>", " ", text)                   # HTML tags
        text = re.sub(r"http\S+|www\.\S+", " ", text)          # URLs
        text = re.sub(r"[^a-zA-Z\s]", " ", text)               # non-alpha
        text = re.sub(r"\s+", " ", text).strip().lower()        # whitespace
        return text

    def fit(self, texts: List[str]) -> "TextPreprocessor":
        cleaned = [self.clean_text(t) for t in texts]
        self.tokenizer.fit_on_texts(cleaned)
        self._fitted = True

        # Vocabulary stats
        word_counts = self.tokenizer.word_counts
        self.vocab_size = min(self.max_vocab, len(word_counts) + 1)
        print(f"  Vocabulary: {len(word_counts):,} unique words → "
              f"capped at {self.vocab_size:,}")
        return self

    def transform(self, texts: List[str]) -> np.ndarray:
        assert self._fitted, "Call fit() first."
        cleaned = [self.clean_text(t) for t in texts]
        seqs = self.tokenizer.texts_to_sequences(cleaned)
        padded = pad_sequences(
            seqs, maxlen=self.max_len,
            padding=self.padding, truncating=self.truncating,
        )
        return padded

    def fit_transform(self, texts: List[str]) -> np.ndarray:
        self.fit(texts)
        return self.transform(texts)

    @property
    def word_index(self) -> dict:
        return self.tokenizer.word_index


def load_imdb_dataset(
    max_vocab: int = 25_000,
    max_len: int = 256,
) -> Tuple[dict, TextPreprocessor]:
    """
    Load IMDB via keras.datasets and reprocess from raw text
    (avoids pre-tokenized integer sequences for real-world fidelity).
    """
    print("=" * 70)
    print("LOADING IMDB DATASET")
    print("=" * 70)

    # Load raw integer sequences and reverse-map to text
    word_index = keras.datasets.imdb.get_word_index()
    reverse_index = {v + 3: k for k, v in word_index.items()}
    reverse_index[0] = "<PAD>"
    reverse_index[1] = "<START>"
    reverse_index[2] = "<OOV>"
    reverse_index[3] = "<UNUSED>"

    (X_train_idx, y_train), (X_test_idx, y_test) = keras.datasets.imdb.load_data(
        num_words=max_vocab
    )

    # Reconstruct raw text
    def decode(seq):
        return " ".join(reverse_index.get(i, "?") for i in seq)

    texts_train = [decode(s) for s in X_train_idx]
    texts_test  = [decode(s) for s in X_test_idx]

    print(f"  Train: {len(texts_train):,} reviews")
    print(f"  Test:  {len(texts_test):,} reviews")
    print(f"  Class balance — Pos: {y_train.sum():,}  Neg: {(1-y_train).sum():,}")

    # Preprocess
    preprocessor = TextPreprocessor(max_vocab=max_vocab, max_len=max_len)
    X_train = preprocessor.fit_transform(texts_train)
    X_test  = preprocessor.transform(texts_test)

    # Validation split from training set
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train,
    )
    print(f"  After split — Train: {len(X_train):,} | Val: {len(X_val):,}"
          f" | Test: {len(X_test):,}")

    data = {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "texts_test": texts_test,
    }
    return data, preprocessor


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  2. GLOVE EMBEDDING MATRIX                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def load_glove_embeddings(
    glove_path: str,
    word_index: dict,
    vocab_size: int,
    embed_dim: int = 100,
) -> np.ndarray:
    """
    Build an embedding matrix from pre-trained GloVe vectors.
    Words not found in GloVe get random initialisation.

    Download GloVe from: https://nlp.stanford.edu/projects/glove/
    Expected file: glove.6B.100d.txt
    """
    print(f"  Loading GloVe from {glove_path}...")
    embeddings_index = {}
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = vector

    print(f"  GloVe vectors loaded: {len(embeddings_index):,}")

    # Build matrix
    embedding_matrix = np.random.normal(0, 0.05, (vocab_size, embed_dim)).astype("float32")
    found = 0
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        vec = embeddings_index.get(word)
        if vec is not None and len(vec) == embed_dim:
            embedding_matrix[idx] = vec
            found += 1

    coverage = found / min(len(word_index), vocab_size) * 100
    print(f"  Embedding coverage: {found:,}/{min(len(word_index), vocab_size):,}"
          f" ({coverage:.1f}%)")
    return embedding_matrix


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  3. CUSTOM ATTENTION LAYERS                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class SelfAttention(layers.Layer):
    """
    Scaled dot-product self-attention for sequence classification.

    Given H ∈ ℝ^{T×d} (BiLSTM outputs):
        Q = H·W_q,   K = H·W_k,   V = H·W_v
        Attention(Q,K,V) = softmax(QK^T / √d_k) · V

    Returns context vector (weighted sum) for classification.
    """

    def __init__(self, units: int = 64, return_attention: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_attention = return_attention

    def build(self, input_shape):
        d = input_shape[-1]
        self.W_q = self.add_weight("W_q", (d, self.units), initializer="glorot_uniform")
        self.W_k = self.add_weight("W_k", (d, self.units), initializer="glorot_uniform")
        self.W_v = self.add_weight("W_v", (d, self.units), initializer="glorot_uniform")

    def call(self, x):
        Q = tf.matmul(x, self.W_q)   # (B, T, units)
        K_mat = tf.matmul(x, self.W_k)
        V = tf.matmul(x, self.W_v)

        scale = tf.math.sqrt(tf.cast(self.units, tf.float32))
        scores = tf.matmul(Q, K_mat, transpose_b=True) / scale   # (B, T, T)
        weights = tf.nn.softmax(scores, axis=-1)

        context = tf.matmul(weights, V)                           # (B, T, units)
        context = tf.reduce_mean(context, axis=1)                 # (B, units) — pool

        if self.return_attention:
            return context, weights
        return context

    def get_config(self):
        return {**super().get_config(), "units": self.units,
                "return_attention": self.return_attention}


class MultiHeadSelfAttention(layers.Layer):
    """
    Multi-head self-attention (Transformer-style) applied to BiLSTM outputs.

    Splits Q, K, V into `n_heads` parallel attention heads, concatenates,
    and projects back. Adds residual connection + layer normalisation.
    """

    def __init__(self, d_model: int = 128, n_heads: int = 4, **kwargs):
        super().__init__(**kwargs)
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.depth = d_model // n_heads

    def build(self, input_shape):
        self.Wq = layers.Dense(self.d_model, name="query")
        self.Wk = layers.Dense(self.d_model, name="key")
        self.Wv = layers.Dense(self.d_model, name="value")
        self.dense_out = layers.Dense(self.d_model, name="out_proj")
        self.layernorm = layers.LayerNormalization(epsilon=1e-6)
        self.projection = layers.Dense(input_shape[-1], name="residual_proj")

    def _split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.n_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])   # (B, heads, T, depth)

    def call(self, x):
        B = tf.shape(x)[0]

        Q = self._split_heads(self.Wq(x), B)
        K_split = self._split_heads(self.Wk(x), B)
        V = self._split_heads(self.Wv(x), B)

        scale = tf.math.sqrt(tf.cast(self.depth, tf.float32))
        scores = tf.matmul(Q, K_split, transpose_b=True) / scale
        weights = tf.nn.softmax(scores, axis=-1)

        attended = tf.matmul(weights, V)                                    # (B, heads, T, depth)
        attended = tf.transpose(attended, perm=[0, 2, 1, 3])               # (B, T, heads, depth)
        concatenated = tf.reshape(attended, (B, -1, self.d_model))         # (B, T, d_model)

        output = self.dense_out(concatenated)                               # (B, T, d_model)

        # Residual connection (project x if dims mismatch)
        residual = self.projection(x) if x.shape[-1] != output.shape[-1] else x
        output = self.layernorm(output + residual)

        return tf.reduce_mean(output, axis=1)                              # (B, d_model)

    def get_config(self):
        return {**super().get_config(), "d_model": self.d_model,
                "n_heads": self.n_heads}


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  4. MODEL ARCHITECTURES                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _embedding_block(
    vocab_size: int, embed_dim: int, max_len: int,
    embedding_matrix: Optional[np.ndarray] = None,
    spatial_dropout: float = 0.3,
) -> Tuple[layers.Input, layers.Layer]:
    """Shared embedding block with optional pre-trained weights."""
    inp = layers.Input(shape=(max_len,), name="input_tokens")

    if embedding_matrix is not None:
        emb = layers.Embedding(
            vocab_size, embed_dim, input_length=max_len,
            weights=[embedding_matrix], trainable=False,    # freeze GloVe
            mask_zero=False, name="pretrained_embedding",
        )(inp)
    else:
        emb = layers.Embedding(
            vocab_size, embed_dim, input_length=max_len,
            mask_zero=False, name="trainable_embedding",
        )(inp)

    emb = layers.SpatialDropout1D(spatial_dropout, name="spatial_drop")(emb)
    return inp, emb


def build_bilstm_baseline(
    vocab_size: int, embed_dim: int, max_len: int,
    lstm_units: int = 128, dropout: float = 0.3,
    embedding_matrix: Optional[np.ndarray] = None,
) -> keras.Model:
    """
    Architecture A — BiLSTM Baseline
    ─────────────────────────────────
    Embedding → SpatialDropout → BiLSTM → GlobalMaxPool ⊕ GlobalAvgPool
    → Dense → Dropout → Sigmoid
    """
    inp, emb = _embedding_block(vocab_size, embed_dim, max_len, embedding_matrix)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True,
                    dropout=dropout, recurrent_dropout=dropout),
        name="bilstm",
    )(emb)

    # Dual pooling — captures both max-activation and average semantics
    avg_pool = layers.GlobalAveragePooling1D(name="avg_pool")(x)
    max_pool = layers.GlobalMaxPooling1D(name="max_pool")(x)
    x = layers.Concatenate(name="pool_concat")([avg_pool, max_pool])

    x = layers.Dense(64, activation="relu", name="fc")(x)
    x = layers.Dropout(dropout, name="drop")(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inp, out, name="BiLSTM_Baseline")


def build_stacked_bilstm(
    vocab_size: int, embed_dim: int, max_len: int,
    units: List[int] = [128, 64], dropout: float = 0.3,
    embedding_matrix: Optional[np.ndarray] = None,
) -> keras.Model:
    """
    Architecture B — Stacked BiLSTM with Residual Connections
    ──────────────────────────────────────────────────────────
    Embedding → [BiLSTM → LayerNorm → Residual] × N
    → GlobalMaxPool ⊕ GlobalAvgPool → Dense → Sigmoid
    """
    inp, emb = _embedding_block(vocab_size, embed_dim, max_len, embedding_matrix)

    x = emb
    for i, u in enumerate(units):
        bilstm_out = layers.Bidirectional(
            layers.LSTM(u, return_sequences=True,
                        dropout=dropout, recurrent_dropout=dropout),
            name=f"bilstm_{i+1}",
        )(x)
        bilstm_out = layers.LayerNormalization(name=f"ln_{i+1}")(bilstm_out)

        # Residual connection (project dims if mismatch)
        if x.shape[-1] != bilstm_out.shape[-1]:
            x = layers.Dense(bilstm_out.shape[-1], name=f"res_proj_{i+1}")(x)
        x = layers.Add(name=f"residual_{i+1}")([x, bilstm_out])

    avg_pool = layers.GlobalAveragePooling1D(name="avg_pool")(x)
    max_pool = layers.GlobalMaxPooling1D(name="max_pool")(x)
    x = layers.Concatenate()([avg_pool, max_pool])

    x = layers.Dense(64, activation="relu", name="fc1")(x)
    x = layers.Dropout(dropout, name="drop")(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inp, out, name="Stacked_BiLSTM_Residual")


def build_bilstm_attention(
    vocab_size: int, embed_dim: int, max_len: int,
    lstm_units: int = 128, n_heads: int = 4, dropout: float = 0.3,
    embedding_matrix: Optional[np.ndarray] = None,
) -> keras.Model:
    """
    Architecture C — BiLSTM + Multi-Head Self-Attention
    ────────────────────────────────────────────────────
    Embedding → BiLSTM (return_sequences) → MultiHeadSelfAttention
    → Dense → Dropout → Sigmoid

    The attention mechanism learns which tokens (words) contribute
    most to the overall sentiment polarity.
    """
    inp, emb = _embedding_block(vocab_size, embed_dim, max_len, embedding_matrix)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True,
                    dropout=dropout, recurrent_dropout=dropout),
        name="bilstm",
    )(emb)

    d_model = lstm_units * 2   # BiLSTM doubles the dim
    # Ensure d_model is divisible by n_heads
    d_model = (d_model // n_heads) * n_heads

    x = MultiHeadSelfAttention(d_model=d_model, n_heads=n_heads, name="mha")(x)

    x = layers.Dense(64, activation="relu", name="fc")(x)
    x = layers.Dropout(dropout, name="drop")(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inp, out, name="BiLSTM_MultiHeadAttn")


def build_cnn_bilstm(
    vocab_size: int, embed_dim: int, max_len: int,
    n_filters: int = 128, kernel_sizes: List[int] = [2, 3, 4, 5],
    lstm_units: int = 128, dropout: float = 0.3,
    embedding_matrix: Optional[np.ndarray] = None,
) -> keras.Model:
    """
    Architecture D — CNN + BiLSTM Hybrid
    ─────────────────────────────────────
    Embedding → Parallel Conv1D (multi-kernel) → Concat
    → BiLSTM → GlobalMaxPool → Dense → Sigmoid

    Conv1D captures n-gram features (bigrams, trigrams, etc.)
    which are then contextualised by the BiLSTM.
    """
    inp, emb = _embedding_block(vocab_size, embed_dim, max_len, embedding_matrix)

    # Parallel convolutions for multi-scale n-gram extraction
    conv_outputs = []
    for ks in kernel_sizes:
        c = layers.Conv1D(n_filters, ks, padding="same",
                          activation="relu", name=f"conv_k{ks}")(emb)
        c = layers.BatchNormalization(name=f"bn_k{ks}")(c)
        conv_outputs.append(c)

    x = layers.Concatenate(name="conv_concat")(conv_outputs)
    x = layers.MaxPooling1D(pool_size=2, name="pool")(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=False,
                    dropout=dropout, recurrent_dropout=dropout),
        name="bilstm",
    )(x)

    x = layers.Dense(64, activation="relu", name="fc")(x)
    x = layers.Dropout(dropout, name="drop")(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inp, out, name="CNN_BiLSTM_Hybrid")


# Model registry
SENTIMENT_MODELS: Dict[str, callable] = {
    "bilstm_baseline":  build_bilstm_baseline,
    "stacked_bilstm":   build_stacked_bilstm,
    "bilstm_attention":  build_bilstm_attention,
    "cnn_bilstm":        build_cnn_bilstm,
}


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  5. TRAINING PIPELINE                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def compile_and_train(
    model: keras.Model,
    data: dict,
    epochs: int = 30,
    batch_size: int = 128,
    lr: float = 1e-3,
    patience: int = 5,
    label_smoothing: float = 0.05,
) -> callbacks.History:
    """
    Compile with:
      • Binary crossentropy + label smoothing
      • Adam with gradient clipping
      • LR reduction on plateau + early stopping
    """
    loss_fn = keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss=loss_fn,
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    cb = [
        callbacks.EarlyStopping(
            monitor="val_auc", patience=patience, mode="max",
            restore_best_weights=True, verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3,
            min_lr=1e-7, verbose=1,
        ),
    ]

    # Class weights (handles imbalance)
    n_pos = data["y_train"].sum()
    n_neg = len(data["y_train"]) - n_pos
    class_weight = {0: len(data["y_train"]) / (2 * n_neg),
                    1: len(data["y_train"]) / (2 * n_pos)}

    history = model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=epochs, batch_size=batch_size,
        class_weight=class_weight, callbacks=cb, verbose=1,
    )
    return history


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  6. EVALUATION & METRICS                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def evaluate_model(
    model: keras.Model, data: dict, model_name: str = "",
    threshold: float = 0.5,
) -> Dict[str, float]:
    """Full evaluation: classification report, AUC-ROC, optimal threshold."""
    y_true = data["y_test"]
    y_prob = model.predict(data["X_test"], verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{'─'*50}")
    print(f"  EVALUATION: {model_name}")
    print(f"{'─'*50}")
    print(classification_report(
        y_true, y_pred, target_names=["Negative", "Positive"], digits=4,
    ))

    auc = roc_auc_score(y_true, y_prob)
    f1  = f1_score(y_true, y_pred)

    # Find optimal threshold via Youden's J statistic
    fpr, tpr, thresholds_roc = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds_roc[optimal_idx]

    print(f"  AUC-ROC:            {auc:.4f}")
    print(f"  F1 Score:           {f1:.4f}")
    print(f"  Optimal Threshold:  {optimal_threshold:.4f}")
    print(f"  (Youden's J = {j_scores[optimal_idx]:.4f})")

    return {
        "accuracy": np.mean(y_pred == y_true),
        "f1": f1, "auc_roc": auc,
        "optimal_threshold": optimal_threshold,
        "y_prob": y_prob,
    }


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  7. VISUALISATION                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def plot_training_history(history: callbacks.History, title: str = ""):
    """Training curves: loss, accuracy, AUC."""
    metrics = ["loss", "accuracy", "auc"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    for ax, m in zip(axes, metrics):
        ax.plot(history.history[m], label="Train", linewidth=1.5)
        ax.plot(history.history[f"val_{m}"], label="Val", linewidth=1.5)
        ax.set_title(f"{title} — {m.upper()}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("sentiment_training_curves.png", dpi=150)
    plt.close()
    print("  ✓ Saved: sentiment_training_curves.png")


def plot_confusion_matrix(y_true, y_prob, threshold=0.5, title=""):
    """Confusion matrix heatmap."""
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
    plt.colorbar(im, ax=ax)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}",
                    ha="center", va="center", fontsize=14,
                    color="white" if cm[i,j] > cm.max()/2 else "black")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Negative", "Positive"])
    ax.set_yticklabels(["Negative", "Positive"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {title}")
    plt.tight_layout()
    plt.savefig("sentiment_confusion_matrix.png", dpi=150)
    plt.close()
    print("  ✓ Saved: sentiment_confusion_matrix.png")


def plot_roc_pr_curves(y_true, y_prob, title=""):
    """ROC curve + Precision-Recall curve side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax1.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.4f}")
    ax1.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax1.fill_between(fpr, tpr, alpha=0.15)
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title(f"ROC Curve — {title}")
    ax1.legend(loc="lower right")
    ax1.grid(True, alpha=0.3)

    # Precision-Recall
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ax2.plot(recall, precision, linewidth=2, color="coral")
    ax2.fill_between(recall, precision, alpha=0.15, color="coral")
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title(f"Precision-Recall Curve — {title}")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("sentiment_roc_pr.png", dpi=150)
    plt.close()
    print("  ✓ Saved: sentiment_roc_pr.png")


def plot_model_comparison(results: Dict[str, Dict[str, float]]):
    """Bar chart comparing all architectures."""
    metrics_to_plot = ["accuracy", "f1", "auc_roc"]
    labels = list(results.keys())
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = plt.cm.Set2(np.arange(len(labels)))

    for ax, m in zip(axes, metrics_to_plot):
        vals = [results[l][m] for l in labels]
        bars = ax.barh(labels, vals, color=colors)
        ax.set_title(m.upper().replace("_", " "), fontsize=12)
        ax.set_xlim(0.5, 1.0)
        ax.grid(True, axis="x", alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(v + 0.002, bar.get_y() + bar.get_height()/2,
                    f"{v:.4f}", va="center", fontsize=10)

    plt.tight_layout()
    plt.savefig("sentiment_model_comparison.png", dpi=150)
    plt.close()
    print("  ✓ Saved: sentiment_model_comparison.png")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  8. INFERENCE API                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class SentimentPredictor:
    """
    Production-ready inference wrapper.

    Usage:
        predictor = SentimentPredictor(model, preprocessor)
        result = predictor.predict("This movie was absolutely fantastic!")
        # → {'text': '...', 'sentiment': 'POSITIVE', 'confidence': 0.97,
        #    'probability': 0.9713}
    """

    def __init__(
        self,
        model: keras.Model,
        preprocessor: TextPreprocessor,
        threshold: float = 0.5,
    ):
        self.model = model
        self.preprocessor = preprocessor
        self.threshold = threshold

    def predict(self, text: str) -> Dict:
        tokens = self.preprocessor.transform([text])
        prob = float(self.model.predict(tokens, verbose=0)[0, 0])
        sentiment = "POSITIVE" if prob >= self.threshold else "NEGATIVE"
        confidence = prob if sentiment == "POSITIVE" else 1 - prob

        return {
            "text": text[:100] + ("..." if len(text) > 100 else ""),
            "sentiment": sentiment,
            "confidence": round(confidence, 4),
            "probability": round(prob, 4),
        }

    def predict_batch(self, texts: List[str]) -> List[Dict]:
        return [self.predict(t) for t in texts]


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  9. MAIN EXPERIMENT                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def run_sentiment_experiment(
    architectures: Optional[List[str]] = None,
    epochs: int = 20,
    batch_size: int = 128,
    max_vocab: int = 25_000,
    max_len: int = 256,
    embed_dim: int = 128,
    glove_path: Optional[str] = None,
) -> None:
    """End-to-end: load → preprocess → train → evaluate → visualise → infer."""

    if architectures is None:
        architectures = list(SENTIMENT_MODELS.keys())

    # ── Data ──
    data, preprocessor = load_imdb_dataset(max_vocab=max_vocab, max_len=max_len)
    vocab_size = preprocessor.vocab_size

    # Optional GloVe
    embedding_matrix = None
    if glove_path and os.path.exists(glove_path):
        embedding_matrix = load_glove_embeddings(
            glove_path, preprocessor.word_index, vocab_size, embed_dim,
        )
        embed_dim = embedding_matrix.shape[1]

    all_results = {}

    # ── Train & evaluate each architecture ──
    for arch_name in architectures:
        print("\n" + "=" * 70)
        print(f"  ARCHITECTURE: {arch_name.upper()}")
        print("=" * 70)

        build_fn = SENTIMENT_MODELS[arch_name]
        model = build_fn(
            vocab_size=vocab_size, embed_dim=embed_dim, max_len=max_len,
            embedding_matrix=embedding_matrix,
        )
        model.summary(print_fn=lambda s: print(f"    {s}"), line_length=90)

        history = compile_and_train(
            model, data, epochs=epochs, batch_size=batch_size,
        )

        plot_training_history(history, title=arch_name)

        result = evaluate_model(model, data, model_name=arch_name)
        all_results[arch_name] = result

        # Detailed plots for each model
        plot_confusion_matrix(data["y_test"], result["y_prob"], title=arch_name)
        plot_roc_pr_curves(data["y_test"], result["y_prob"], title=arch_name)

    # ── Comparison ──
    comparison = {k: {m: v[m] for m in ["accuracy", "f1", "auc_roc"]}
                  for k, v in all_results.items()}
    plot_model_comparison(comparison)

    print("\n" + "=" * 70)
    print("  FINAL COMPARISON")
    print("=" * 70)
    df = pd.DataFrame(comparison).T
    print(df.to_string(float_format="{:.4f}".format))

    best = df["auc_roc"].idxmax()
    print(f"\n  🏆 Best model by AUC-ROC: {best} ({df.loc[best, 'auc_roc']:.4f})")

    # ── Live Inference Demo ──
    print("\n" + "=" * 70)
    print("  INFERENCE DEMO")
    print("=" * 70)

    # Rebuild best model for inference
    best_model = SENTIMENT_MODELS[best](
        vocab_size=vocab_size, embed_dim=embed_dim, max_len=max_len,
        embedding_matrix=embedding_matrix,
    )
    best_model.compile(optimizer="adam", loss="binary_crossentropy")
    best_model.fit(
        data["X_train"], data["y_train"],
        validation_data=(data["X_val"], data["y_val"]),
        epochs=epochs, batch_size=batch_size, verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    )

    predictor = SentimentPredictor(best_model, preprocessor)

    demo_texts = [
        "This movie was absolutely brilliant! The acting was superb and the plot kept me engaged throughout.",
        "Terrible waste of time. The dialogue was cringeworthy and the plot made no sense at all.",
        "It was okay, nothing special. Had some good moments but overall fairly forgettable.",
        "A masterpiece of modern cinema. Every scene was carefully crafted with stunning cinematography.",
        "I walked out after 30 minutes. Worst film I have ever seen, complete garbage.",
        "The film started slow but built into something truly memorable by the final act.",
    ]

    for text in demo_texts:
        result = predictor.predict(text)
        emoji = "🟢" if result["sentiment"] == "POSITIVE" else "🔴"
        print(f"\n  {emoji} {result['sentiment']} (conf: {result['confidence']:.2%})")
        print(f"     \"{result['text']}\"")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ENTRY POINT                                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

if __name__ == "__main__":
    run_sentiment_experiment(
        architectures=["bilstm_baseline", "stacked_bilstm",
                        "bilstm_attention", "cnn_bilstm"],
        epochs=20,
        batch_size=128,
        max_vocab=25_000,
        max_len=256,
        embed_dim=128,
        glove_path=None,         # Set to "glove.6B.100d.txt" if available
    )


LOADING IMDB DATASET
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Train: 25,000 reviews
  Test:  25,000 reviews
  Class balance — Pos: 12,500  Neg: 12,500
  Vocabulary: 23,466 unique words → capped at 23,467
  After split — Train: 21,250 | Val: 3,750 | Test: 25,000

  ARCHITECTURE: BILSTM_BASELINE


    Model: "BiLSTM_Baseline"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_tokens             │ (None, 256)          │            0 │ -                     │
│ (InputLayer)             │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ trainable_embedding      │ (None, 256, 128)     │    3,003,776 │ input_tokens[0][0]    │
│ (Embedding)              │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ spatial_drop             │ (None, 256, 128)     │            0 │ trainable_embedding[… │
│ (SpatialDropout1D)       │                      │          